In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))          # notebooks/ -> repo root
from config import directories

import pandas as pd
from sqlalchemy import create_engine

DB_PATH = directories.INTERIM_DATA / "panchayat_1.duckdb"
engine = create_engine(f"duckdb:///{DB_PATH}")

def q(sql):
    """Run SQL and return a DataFrame."""
    return pd.read_sql(sql, engine)

# confirm what's there
print(q("SHOW TABLES").to_string(index=False))

                      name
            activity_asset
activity_community_service
       activity_delegation
      activity_expenditure
             activity_fund
             activity_nsap
         activity_training
          activity_voucher
            gram_panchayat
                      plan
          planned_activity
                   voucher


In [2]:
for t in ["gram_panchayat", "plan", "planned_activity", "activity_expenditure",
          "voucher", "activity_voucher"]:
    print(f"{t:24} {q(f'SELECT count(*) FROM {t}').iloc[0,0]:>7}")

gram_panchayat                20
plan                         204
planned_activity           12704
activity_expenditure       12730
voucher                    12440
activity_voucher            5976


In [3]:
# Q1: "How many activities did each gram panchayat plan, and what did they
#      budget, year by year?"
q("""
SELECT g.gp_name, a.fiscal_year,
       count(*) AS activities,
       sum(a.total_cost) AS planned_cost
FROM planned_activity a
JOIN gram_panchayat g USING (gp_lgd_code)
GROUP BY 1,2 ORDER BY 1,2
""")

,gp_name,fiscal_year,activities,planned_cost
0,Andhrua,2020-2021,16,3947654.0
1,Andhrua,2021-2022,27,3636707.0
2,Andhrua,2022-2023,35,6000000.0
3,Andhrua,2023-2024,211,4069134.0
4,Andhrua,2024-2025,118,3518192.0
...,...,...,...,...
115,Sharagada,2021-2022,30,9702187.0
116,Sharagada,2022-2023,29,8130750.0
117,Sharagada,2023-2024,433,5805546.0
118,Sharagada,2024-2025,276,4825633.0


In [4]:
# Q2: "For Andhrua, how much was planned versus actually spent each year,
#      and what's the utilisation rate?"
q("""
SELECT a.fiscal_year,
       count(*) AS activities,
       sum(a.total_cost) AS planned,
       sum(e.total_expenditure) AS spent,
       round(100.0*sum(e.total_expenditure)/nullif(sum(a.total_cost),0),1) AS util_pct
FROM planned_activity a
LEFT JOIN activity_expenditure e USING (activity_code)
WHERE a.gp_lgd_code='119598'
GROUP BY 1 ORDER BY 1
""")

,fiscal_year,activities,planned,spent,util_pct
0,2020-2021,16,3947654.0,3432694.00,87.0
1,2021-2022,27,3636707.0,1661841.00,45.7
2,2022-2023,35,6000000.0,3107156.00,51.8
3,2023-2024,211,4069134.0,2401719.81,59.0
4,2024-2025,118,3518192.0,2649400.00,75.3
5,2025-2026,86,3888192.0,1601731.00,41.2


In [5]:
# Q1: "How many activities did Andhrua plan each year, and what did it budget?"
q("""
SELECT a.fiscal_year,
       count(*) AS activities,
       sum(a.total_cost) AS planned_cost
FROM planned_activity a
WHERE a.gp_lgd_code = '119598'
GROUP BY 1 ORDER BY 1
""")

,fiscal_year,activities,planned_cost
0,2020-2021,16,3947654.0
1,2021-2022,27,3636707.0
2,2022-2023,35,6000000.0
3,2023-2024,211,4069134.0
4,2024-2025,118,3518192.0
5,2025-2026,86,3888192.0


In [6]:
# Q2: "For Andhrua, how much was planned versus actually spent each year,
#      and what's the utilisation rate?"
q("""
SELECT a.fiscal_year,
       count(*) AS activities,
       sum(a.total_cost) AS planned,
       sum(e.total_expenditure) AS spent,
       round(100.0*sum(e.total_expenditure)/nullif(sum(a.total_cost),0),1) AS util_pct
FROM planned_activity a
LEFT JOIN activity_expenditure e USING (activity_code)
WHERE a.gp_lgd_code = '119598'
GROUP BY 1 ORDER BY 1
""")

,fiscal_year,activities,planned,spent,util_pct
0,2020-2021,16,3947654.0,3432694.00,87.0
1,2021-2022,27,3636707.0,1661841.00,45.7
2,2022-2023,35,6000000.0,3107156.00,51.8
3,2023-2024,211,4069134.0,2401719.81,59.0
4,2024-2025,118,3518192.0,2649400.00,75.3
5,2025-2026,86,3888192.0,1601731.00,41.2


In [13]:
# Q3: "What were Andhrua's total receipts and payments in each financial year?"
q("""
SELECT fiscal_year, direction, count(*) AS vouchers, sum(amount) AS total
FROM voucher
WHERE gp_lgd_code = '119598'
GROUP BY 1,2 ORDER BY 1,2
""")

,fiscal_year,direction,vouchers,total
0,2020-2021,payment,92,14825752.70
1,2020-2021,receipt,63,15158497.00
2,2021-2022,payment,104,13703659.65
3,2021-2022,receipt,60,13042697.00
4,2022-2023,payment,172,7737794.71
5,2022-2023,receipt,94,13920605.60
6,2023-2024,payment,121,16004791.40
7,2023-2024,receipt,62,14118968.30
8,2024-2025,payment,131,7123909.00
9,2024-2025,receipt,35,6421728.28


In [8]:
# Q4: "What are Andhrua's ten largest single payments?"
q("""
SELECT v.fiscal_year, v.voucher_no, v.type, v.date, v.amount
FROM voucher v
WHERE v.gp_lgd_code = '119598' AND v.direction = 'payment'
ORDER BY v.amount DESC
LIMIT 10
""")

,fiscal_year,voucher_no,type,date,amount
0,2025-2026,OWN/2025-26/P/1,Expenditures,2026-03-30,2553765.0
1,2020-2021,NOAPS/2020-21/P/4,Expenditures,2021-01-08,2212000.0
2,2024-2025,OWN/2024-25/P/2,Expenditures,2025-03-26,1897811.0
3,2020-2021,NOAPS/2020-21/P/3,Expenditures,2020-09-20,1885600.0
4,2023-2024,PDS/2023-24/P/9,Expenditures,2024-03-23,1874628.9
5,2020-2021,PDS/2020-21/P/4,Expenditures,2020-06-30,1697000.0
6,2023-2024,FFC/2023-24/P/12,Expenditures,2023-07-04,1690236.0
7,2021-2022,PDS/2021-22/P/11,Expenditures,2022-03-02,1669000.0
8,2020-2021,NOAPS/2020-21/P/2,Expenditures,2020-08-25,1322500.0
9,2021-2022,NOAPS/2021-22/P/5,Expenditures,2022-03-01,1320500.0


In [9]:
# Q6: "How much of Andhrua's funding is earmarked for General, SC, and ST
#      categories each year?"
q("""
SELECT a.fiscal_year,
       sum(f.fund_tied_general + f.fund_untied_general) AS general,
       sum(f.fund_tied_sc + f.fund_untied_sc)           AS sc,
       sum(f.fund_tied_st + f.fund_untied_st)           AS st,
       sum(f.fund_amount_total)                         AS total
FROM activity_fund f
JOIN planned_activity a USING (activity_code)
WHERE a.gp_lgd_code = '119598'
GROUP BY 1 ORDER BY 1
""")

,fiscal_year,general,sc,st,total
0,2020-2021,3947654.0,0.0,0.0,3947654.0
1,2021-2022,3636707.0,0.0,0.0,3636707.0
2,2022-2023,6000000.0,0.0,0.0,6000000.0
3,2023-2024,4069134.0,0.0,0.0,4069134.0
4,2024-2025,3518192.0,0.0,0.0,3518192.0
5,2025-2026,3888192.0,0.0,0.0,3888192.0


In [10]:
# Q7: "Show me Andhrua's 2025-26 activities traced all the way to the
#      individual vouchers that paid for them."
q("""
SELECT a.activity_code, a.activity_name, a.total_cost,
       e.total_expenditure, av.voucher_no, av.voucher_cost, v.date
FROM planned_activity a
JOIN activity_expenditure e USING (activity_code)
JOIN activity_voucher av USING (expenditure_id)
LEFT JOIN voucher v USING (voucher_pk)
WHERE a.gp_lgd_code = '119598' AND a.fiscal_year = '2025-2026'
ORDER BY av.voucher_cost DESC
LIMIT 15
""")

,activity_code,activity_name,total_cost,total_expenditure,voucher_no,voucher_cost,date
0,122408190,Operation & maintenance of community sanitary ...,201672.0,201672.0,XVFC/2025-26/P/181,180000.0,2026-03-23
1,122778951,Construction of Paver Block,200000.0,196489.0,5THSFC/2025-26/P/55,165414.0,2026-03-10
2,127377934,Construction of roads,200000.0,196480.0,5THSFC/2025-26/P/34,160000.0,2026-02-21
3,122463524,Construction of Paver Block,150896.0,148270.0,XVFC/2025-26/P/170,120000.0,2026-03-10
4,122511991,Construction of Paver Block,150000.0,147385.0,XVFC/2025-26/P/157,120000.0,2026-02-20
5,122785257,Construction of roads,150000.0,147385.0,5THSFC/2025-26/P/37,120000.0,2026-02-21
6,127378005,Construction of roads,150000.0,147384.0,5THSFC/2025-26/P/43,120000.0,2026-02-21
7,122432907,Piped/tap water supply to HH,100672.0,100672.0,XVFC/2025-26/P/184,90000.0,2026-03-27
8,122494657,Construction of roads,100000.0,98251.0,XVFC/2025-26/P/154,80000.0,2026-02-20
9,122790071,Construction of Paver Block,85952.0,84771.0,5THSFC/2025-26/P/40,70000.0,2026-02-21


In [1]:
#open updated database
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from config import directories

import pandas as pd
from sqlalchemy import create_engine

DB_PATH = directories.INTERIM_DATA / "panchayat_1.duckdb"
engine = create_engine(f"duckdb:///{DB_PATH}")
def q(sql): return pd.read_sql(sql, engine)

tables = q("SHOW TABLES")
print(f"{len(tables)} tables (expect 19)\n")
print(tables.to_string(index=False))

19 tables (expect 19)

                      name
            activity_asset
activity_community_service
       activity_delegation
      activity_expenditure
             activity_fund
             activity_nsap
         activity_training
          activity_voucher
            admin_approval
     admin_approval_scheme
                  dim_code
            dim_lsdg_theme
        dim_welfare_scheme
            gram_panchayat
         physical_progress
                      plan
          planned_activity
        technical_approval
                   voucher


In [2]:
#check tables are connected 
q("""
SELECT a.activity_code, a.activity_name,
       aa.adm_approval_no, aa.adm_approval_sanction_date,
       aa.work_proposed_cost, ta.tec_approval_cost,
       e.total_expenditure,
       count(pp.row_id) AS progress_records
FROM planned_activity a
JOIN admin_approval aa           USING (activity_code)
LEFT JOIN technical_approval ta  USING (activity_code)
LEFT JOIN activity_expenditure e USING (activity_code)
LEFT JOIN physical_progress pp   USING (activity_code)
WHERE a.gp_lgd_code = '119598'
GROUP BY 1,2,3,4,5,6,7
ORDER BY aa.adm_approval_sanction_date DESC
LIMIT 10
""")

,activity_code,activity_name,adm_approval_no,adm_approval_sanction_date,work_proposed_cost,tec_approval_cost,total_expenditure,progress_records
0,122887323,Construction of roads,27,2026-01-02,150000.0,150000.0,73572.0,3
1,127377934,Construction of roads,1,2025-11-30,200000.0,200000.0,196480.0,3
2,127378005,Construction of roads,2,2025-11-30,150000.0,150000.0,147384.0,3
3,65617848,Technical & administrative expenses,10,2025-11-01,8000.0,8000.0,5900.0,0
4,122433890,Repair of pipe drinking water,5,2025-05-02,140000.0,140000.0,0.0,0
5,102128088,Construction of roads,35,2025-05-02,100000.0,100000.0,98238.0,3
6,101745219,Drainage Construction,28,2025-05-02,270672.0,270672.0,266388.0,4
7,122403174,Operation and Maintenance of drinking/piped wa...,2,2025-05-02,300000.0,300000.0,0.0,0
8,122463524,Construction of Paver Block,26,2025-05-02,150896.0,1501896.0,148270.0,3
9,122778951,Construction of Paver Block,25,2025-05-02,200000.0,200000.0,196489.0,3
